In [1]:
import sqlite3
from pathlib import Path

db = Path('work/demo/catalogue.sqlite').resolve()
query = """
SELECT substr(request_id, 1, 20) AS request,
       json_extract(normalized_json,'$.inputs.green.lineage.item_id') AS scene,
       json_extract(normalized_json,'$.inputs.green.lineage.aoi.bbox') AS bbox,
       json_extract(normalized_json,'$.parameters.threshold') AS threshold,
       json_extract(normalized_json,'$.preparation.mask_policy') AS mask_policy
FROM processing_requests
ORDER BY created_at;
"""
with sqlite3.connect(db.as_uri() + '?mode=ro', uri=True) as c:
    for row in c.execute(query):
        print(row)

('request-cfd9260532b4', 'S2A_T34UDA_20250604T094637_L2A', '[19.91,50.04,19.94,50.06]', 0.0, 'scl')
('request-aa4059258f17', 'S2A_T34UDA_20250604T094637_L2A', '[19.94,50.04,19.97,50.06]', 0.0, 'scl')
('request-8ce2a82b82fa', 'S2C_T34UDA_20250605T095638_L2A', '[19.91,50.04,19.94,50.06]', 0.0, 'scl')
('request-ba94b4a37f39', 'S2C_T34UDA_20250605T095638_L2A', '[19.94,50.04,19.97,50.06]', 0.0, 'scl')
('request-c2469beffbca', 'S2B_T34UDA_20250610T095316_L2A', '[19.91,50.04,19.94,50.06]', 0.0, 'scl')
('request-58e3ed2e7d27', 'S2B_T34UDA_20250610T095316_L2A', '[19.94,50.04,19.97,50.06]', 0.0, 'scl')
('request-d8a75a1e8b21', 'S2A_T34UDA_20250604T094637_L2A', '[19.91,50.04,19.94,50.06]', 0.0, 'nodata_only')
('request-43ad6f1beef1', 'S2C_T34UDA_20250605T095638_L2A', '[19.91,50.04,19.94,50.06]', 0.15, 'scl')


In [2]:
import sqlite3
from pathlib import Path

db = Path('work/demo/catalogue.sqlite').resolve()

def run(sql):
    with sqlite3.connect(db.as_uri() + '?mode=ro', uri=True) as c:
        cur = c.execute(sql)
        print([d[0] for d in cur.description])
        for row in cur.fetchall():
            print(row)

In [3]:
run("""
SELECT assessment_id, attempt_id, assessed_at,
       json_extract(policy_json,'$.minimum_valid_fraction') AS minimum_valid_fraction,
       outcome
FROM scientific_qc_assessments
ORDER BY assessment_id
""")

['assessment_id', 'attempt_id', 'assessed_at', 'minimum_valid_fraction', 'outcome']
(1, 'attempt-2a880d47e4644a9e833c0962e0762790', '2026-11-17T21:07:06.102187Z', 0.5, 'rejected')
(2, 'attempt-05d7f32fea124e909c834430c57d69d0', '2026-11-17T21:07:06.124224Z', 0.5, 'rejected')
(3, 'attempt-6037e0c1fa95456681f9551bb8584896', '2026-11-17T21:07:06.147253Z', 0.5, 'accepted')
(4, 'attempt-42a843c45fae4d16a9a822eae6d32e13', '2026-11-17T21:07:06.168167Z', 0.5, 'accepted')
(5, 'attempt-7d5f9ca6ee504375ba4a72811d895727', '2026-11-17T21:07:06.194204Z', 0.5, 'accepted')
(6, 'attempt-c5bc9b93b6214ff7b79e0dac11bae5b0', '2026-11-17T21:07:06.219375Z', 0.5, 'accepted')
(7, 'attempt-d1a82f482e7f4cb689f954421ac6a866', '2026-11-17T21:39:12.043554Z', 0.5, 'review')
(8, 'attempt-2a880d47e4644a9e833c0962e0762790', '2026-11-17T22:06:44.762987Z', 0.995, 'rejected')
(9, 'attempt-05d7f32fea124e909c834430c57d69d0', '2026-11-17T22:06:44.786840Z', 0.995, 'rejected')
(10, 'attempt-6037e0c1fa95456681f9551bb8584896', '

In [4]:
run ("""

SELECT a.attempt_id, a.status AS execution_status, q.outcome,
       q.assessed_at, q.reason,
       json_extract(q.metrics_json,'$.valid_fraction') AS valid_fraction
FROM processing_attempts AS a
JOIN latest_scientific_qc AS q USING(attempt_id)
WHERE a.status='succeeded' AND q.outcome='rejected'
ORDER BY q.assessment_id;




""")

['attempt_id', 'execution_status', 'outcome', 'assessed_at', 'reason', 'valid_fraction']
('attempt-2a880d47e4644a9e833c0962e0762790', 'succeeded', 'rejected', '2026-11-17T22:09:21.995819Z', 'Usable observation fraction below this assessment policy', 0.4760187824920342)
('attempt-05d7f32fea124e909c834430c57d69d0', 'succeeded', 'rejected', '2026-11-17T22:09:22.015549Z', 'Usable observation fraction below this assessment policy', 0.4853605487702861)
('attempt-6037e0c1fa95456681f9551bb8584896', 'succeeded', 'rejected', '2026-11-17T22:09:22.034229Z', 'Usable observation fraction below this assessment policy', 0.9939627704175751)
('attempt-42a843c45fae4d16a9a822eae6d32e13', 'succeeded', 'rejected', '2026-11-17T22:09:22.057834Z', 'Usable observation fraction below this assessment policy', 0.9915509452902794)
('attempt-7d5f9ca6ee504375ba4a72811d895727', 'succeeded', 'rejected', '2026-11-17T22:09:22.076154Z', 'Usable observation fraction below this assessment policy', 0.9919503605567668)
('atte

In [5]:
run ("""

SELECT a.request_id, a.attempt_id,
       json_extract(r.normalized_json,'$.product.name') AS product,
       o.role, o.artifact_id
FROM processing_attempts AS a
JOIN processing_requests AS r USING(request_id)
JOIN processing_outputs AS o USING(attempt_id)
ORDER BY a.started_at, o.role;


""")

['request_id', 'attempt_id', 'product', 'role', 'artifact_id']
('request-cfd9260532b41af4d476e3d9f7e9a83b729136365adbad8f0fe91aa61ed8e0e1', 'attempt-2a880d47e4644a9e833c0962e0762790', 'mndwi', 'index', 'output-3e0ec8ca07df629085314ae703f7a2611373772c3b5c9bec273f7dd3a52a4121')
('request-cfd9260532b41af4d476e3d9f7e9a83b729136365adbad8f0fe91aa61ed8e0e1', 'attempt-2a880d47e4644a9e833c0962e0762790', 'mndwi', 'valid_mask', 'output-178512523be72feefbf2cc10836c3c2b1b890b13b37abea09dc2c7ad433f8714')
('request-cfd9260532b41af4d476e3d9f7e9a83b729136365adbad8f0fe91aa61ed8e0e1', 'attempt-2a880d47e4644a9e833c0962e0762790', 'mndwi', 'water', 'output-2fa470b9e05798c03bd79fe852270eed56d3ade02e13ee26cc6e937fea00b193')
('request-aa4059258f17503620529c36854a9fc0828cf8072df252b74c20ed6fdff7468f', 'attempt-05d7f32fea124e909c834430c57d69d0', 'mndwi', 'index', 'output-6ae51c881d6c06c7d98834361abdb225030764dad339b103a79ab4b11bc615b2')
('request-aa4059258f17503620529c36854a9fc0828cf8072df252b74c20ed6fdff7468f',

In [6]:
run ("""

SELECT attempt_id, assessment_id, assessed_at, policy_version, outcome,
       json_extract(policy_json,'$.minimum_valid_fraction') AS minimum_valid_fraction,
       json_extract(metrics_json,'$.valid_fraction') AS observed_valid_fraction
FROM scientific_qc_assessments
ORDER BY attempt_id, assessment_id;


""")

['attempt_id', 'assessment_id', 'assessed_at', 'policy_version', 'outcome', 'minimum_valid_fraction', 'observed_valid_fraction']
('attempt-05d7f32fea124e909c834430c57d69d0', 2, '2026-11-17T21:07:06.124224Z', 'coverage-range-v1', 'rejected', 0.5, 0.4853605487702861)
('attempt-05d7f32fea124e909c834430c57d69d0', 9, '2026-11-17T22:06:44.786840Z', 'coverage-range-v1', 'rejected', 0.995, 0.4853605487702861)
('attempt-05d7f32fea124e909c834430c57d69d0', 15, '2026-11-17T22:09:22.015549Z', 'coverage-range-v1', 'rejected', 0.995, 0.4853605487702861)
('attempt-2a880d47e4644a9e833c0962e0762790', 1, '2026-11-17T21:07:06.102187Z', 'coverage-range-v1', 'rejected', 0.5, 0.4760187824920342)
('attempt-2a880d47e4644a9e833c0962e0762790', 8, '2026-11-17T22:06:44.762987Z', 'coverage-range-v1', 'rejected', 0.995, 0.4760187824920342)
('attempt-2a880d47e4644a9e833c0962e0762790', 14, '2026-11-17T22:09:21.995819Z', 'coverage-range-v1', 'rejected', 0.995, 0.4760187824920342)
('attempt-42a843c45fae4d16a9a822eae6d32

In [7]:
run("""

SELECT artifact_id, local_path, status, checked_at
FROM processing_artifact_locations
WHERE status<>'available'
ORDER BY checked_at DESC;



""")

['artifact_id', 'local_path', 'status', 'checked_at']


In [8]:
run(

"""
SELECT m.batch_id, m.member_id, m.planned_request_id
FROM batch_members AS m
WHERE NOT EXISTS (
 SELECT 1 FROM processing_attempts AS a WHERE a.request_id=m.planned_request_id
)
ORDER BY m.batch_id,m.member_id;


"""

    
)

['batch_id', 'member_id', 'planned_request_id']


In [9]:
run("""

SELECT attempt_id, request_id, stage, started_at, log_path
FROM processing_attempts
WHERE status='running';



""")

['attempt_id', 'request_id', 'stage', 'started_at', 'log_path']


In [10]:
run(
    
    """SELECT attempt_id,
       json_extract(receipt_json,'$.metrics.product_metrics.candidate_water_area_m2') AS candidate_water_area_m2,
       json_extract(receipt_json,'$.metrics.product_metrics.observed_area_m2') AS observed_area_m2,
       json_extract(receipt_json,'$.metrics.valid_fraction') AS valid_fraction
FROM processing_attempts
WHERE status='succeeded' AND json_extract(receipt_json,'$.contract')='optical-product-v1';


"""
    
)

['attempt_id', 'candidate_water_area_m2', 'observed_area_m2', 'valid_fraction']
('attempt-2a880d47e4644a9e833c0962e0762790', 130000.0, 2270800.0, 0.4760187824920342)
('attempt-05d7f32fea124e909c834430c57d69d0', 87200.0, 2320800.0, 0.4853605487702861)
('attempt-6037e0c1fa95456681f9551bb8584896', 231200.0, 4741600.0, 0.9939627704175751)
('attempt-42a843c45fae4d16a9a822eae6d32e13', 233600.0, 4741200.0, 0.9915509452902794)
('attempt-7d5f9ca6ee504375ba4a72811d895727', 245600.0, 4732000.0, 0.9919503605567668)
('attempt-c5bc9b93b6214ff7b79e0dac11bae5b0', 260400.0, 4736400.0, 0.9905470972059561)
('attempt-d1a82f482e7f4cb689f954421ac6a866', 233600.0, 4770400.0, 1.0)
('attempt-fe734ef07304477b86fa24b9c823d800', 181600.0, 4741600.0, 0.9939627704175751)
